**Note (release).** The paper's local-model runs were made with this notebook in the development layout.
Data file names and paths were updated for this release: the record files are unpacked to `data/`, as
`eval_banglabbq.py` and `score_banglabbq.py` expect, and the scoring cell names them. Outputs were cleared.
Model-loading and generation cells are unchanged.


# BanglaBBQ — open-model evaluation on Google Colab (free T4 GPU)

Colab port of `evaluation/notebooks/banglabbq_kaggle_eval.ipynb`. Same protocol and output format as
`evaluation/eval_banglabbq.py` (zero-shot, answer letter A/B/C, 3 cyclic option orders, greedy decoding),
so results can be scored together with OpenRouter and Kaggle runs.

**How to run**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked (results are saved there, so a
   disconnect loses nothing). The first time, a file picker asks for `banglabbq_kaggle_upload.zip`;
   it is kept on Drive for later runs.
3. Re-running after a disconnect resumes where it stopped (finished calls are skipped).

Free Colab has **one** T4 (15 GB). Qwen3-4B fits in fp16; Qwen3-8B needs `LOAD_8BIT = True`
(results are then labelled `-int8`, because they are not identical to full precision).

In [ ]:
# ---------- settings: change MODEL_ID for each run ----------
MODEL_ID  = "Qwen/Qwen3-4B"      # fits one T4 in fp16. "Qwen/Qwen3-8B" needs LOAD_8BIT = True
LOAD_8BIT = False                # int8 via bitsandbytes; the model is then labelled "<MODEL_ID>-int8"
LANGS     = ["en", "bn"]
PROMPTS   = [0]          # pilot: [0]; full run: [0, 1, 2]
PERMS     = 3            # cyclic option orders
BATCH     = 16           # lower to 8 if you get CUDA out-of-memory
MAX_NEW   = 8            # answer is one letter
LIMIT     = 200          # smoke test; set to None for the full run

In [ ]:
# ---------- environment: Colab + Google Drive (falls back to a local folder outside Colab) ----------
import os, sys, glob, shutil, subprocess, zipfile
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/banglabbq_eval"
    except Exception as e:
        print("Drive not mounted (", type(e).__name__, ") - results are LOST if the session disconnects.")
        BASE = "/content/banglabbq_eval"
    W = "/content/repo"
else:
    BASE = os.path.abspath(os.environ.get("BANGLABBQ_BASE", "colab_work"))
    W = os.path.join(BASE, "repo")
RES = os.path.join(BASE, "results")
os.makedirs(RES, exist_ok=True)
print("IN_COLAB:", IN_COLAB, "| results ->", RES)

In [ ]:
# ---------- packages (Colab already has torch) ----------
if IN_COLAB:
    pkgs = ["transformers>=4.51", "accelerate"] + (["bitsandbytes"] if LOAD_8BIT else [])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)

In [ ]:
# ---------- GPU check ----------
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 2**30:.1f} GB")
elif IN_COLAB:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all.")
else:
    print("No GPU - local CPU test only")

In [ ]:
# ---------- data + evaluation code (banglabbq_kaggle_upload.zip) ----------
ZIP = os.path.join(BASE, "banglabbq_kaggle_upload.zip")
if not os.path.exists(ZIP):
    if IN_COLAB:
        from google.colab import files
        print("Choose banglabbq_kaggle_upload.zip (written by evaluation/notebooks/make_kaggle_upload.py)")
        up = files.upload()
        name = next(n for n in up if n.endswith(".zip"))
        with open(ZIP, "wb") as f:
            f.write(up[name])
    else:
        shutil.copy(os.environ["BANGLABBQ_ZIP"], ZIP)
os.makedirs(W + "/data", exist_ok=True); os.makedirs(W + "/evaluation", exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        dest = "data" if n.endswith((".jsonl", ".csv")) else "evaluation"
        with z.open(n) as src, open(f"{W}/{dest}/{os.path.basename(n)}", "wb") as dst:
            shutil.copyfileobj(src, dst)
sys.path.insert(0, W + "/evaluation")
import eval_banglabbq as E
print("files ready:", os.listdir(W + "/data"), os.listdir(W + "/evaluation"))

In [ ]:
# ---------- Hugging Face login (only needed for gated models) ----------
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("logged in to Hugging Face")
except Exception as e:
    print("No HF_TOKEN secret - fine for non-gated models such as Qwen.", type(e).__name__)

In [ ]:
# ---------- load the model ----------
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
kw = {}
if torch.cuda.is_available():
    kw["device_map"] = "auto"
    kw["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if LOAD_8BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    kw["torch_dtype"] = torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
model.eval()
MODEL_TAG = MODEL_ID + ("-int8" if LOAD_8BIT else "")
print("loaded", MODEL_TAG, kw.get("torch_dtype"))
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

def chat(prompt):
    msgs = [{"role": "user", "content": prompt}]
    # enable_thinking=False turns off Qwen3's reasoning mode; other templates ignore it
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print(chat(E.build_prompt(E.load_records("en")[0], "en", 0, 0)[0]))

In [ ]:
# ---------- run (resumable: re-running skips finished calls) ----------
import json, re, time
from tqdm.auto import tqdm
FULL_CALLS = {lang: len(E.load_records(lang)) * len(PROMPTS) * PERMS for lang in LANGS}
stats = {}
def run(lang):
    recs = E.load_records(lang)[:LIMIT] if LIMIT else E.load_records(lang)
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", MODEL_TAG)
    out = f"{RES}/{safe}__{lang}__p{''.join(map(str, PROMPTS))}_k{PERMS}.jsonl"
    done = set()
    if os.path.exists(out):
        with open(out, encoding="utf-8") as f:
            for l in f:
                j = json.loads(l); done.add((j["example_id"], j["prompt_id"], j["perm"]))
    jobs = []
    for r in recs:
        for p in PROMPTS:
            for k in range(PERMS):
                if (r["example_id"], p, k) not in done:
                    prompt, order = E.build_prompt(r, lang, p, k)
                    jobs.append((r, p, k, order, chat(prompt)))
    jobs.sort(key=lambda j: len(j[4]))                       # similar lengths per batch = faster
    print(f"{lang}: {len(jobs)} calls to run ({len(done)} already done) -> {out}")
    t0, oos = time.time(), 0
    bar = tqdm(total=len(jobs), desc=lang, unit="call")
    with open(out, "a", encoding="utf-8") as f:
        for i in range(0, len(jobs), BATCH):
            b = jobs[i:i + BATCH]
            enc = tok([j[4] for j in b], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for (r, p, k, order, _), t in zip(b, texts):
                pred = E.parse_answer(t, r, order)
                oos += pred is None
                f.write(json.dumps({
                    "example_id": r["example_id"], "category": r["category"],
                    "question_index": r["question_index"], "lang": lang, "model": MODEL_TAG,
                    "prompt_id": p, "perm": k, "order": order, "seed": 0, "raw": t,
                    "finish": "stop", "error": None, "pred": pred, "usage": None,
                    "provider": "colab-transformers", "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
                }, ensure_ascii=False) + "\n")
            f.flush()
            bar.update(len(b))
            bar.set_postfix(out_of_choice=oos)
    bar.close()
    mins = (time.time() - t0) / 60
    stats[lang] = {"calls": len(jobs), "minutes": mins, "oos": oos}
    print(f"{lang} done in {mins:.1f} min, out-of-choice {oos}/{len(jobs)}")
    return out

for lang in LANGS:
    run(lang)

In [ ]:
# ---------- smoke-test verdict: out-of-choice rate and projected full-run time ----------
for lang, s in stats.items():
    if s["calls"]:
        rate = s["calls"] / max(s["minutes"], 1e-9)
        print(f"{lang}: {s['calls']} calls in {s['minutes']:.1f} min ({rate:.0f} calls/min), "
              f"out-of-choice {s['oos'] / s['calls']:.1%}, "
              f"full run ({FULL_CALLS[lang]} calls) ~ {FULL_CALLS[lang] / rate:.0f} min")

In [ ]:
# ---------- score everything in the results folder ----------
subprocess.run([sys.executable, f"{W}/evaluation/score_banglabbq.py", f"{RES}/*.jsonl",
                "--data-en", f"{W}/data/banglabbq_en.jsonl", "--data-bn", f"{W}/data/banglabbq_bn.jsonl",
                "--out", f"{BASE}/report", "--bootstrap", "1000"], check=True)
print(open(f"{BASE}/report/report.md", encoding="utf-8").read())

In [ ]:
# ---------- pack results for download (also stays on Drive: MyDrive/banglabbq_eval/) ----------
pack = os.path.join(BASE, "pack")
for d in ["results", "report"]:
    if os.path.exists(f"{BASE}/{d}"):
        shutil.copytree(f"{BASE}/{d}", f"{pack}/{d}", dirs_exist_ok=True)
zpath = shutil.make_archive(f"{BASE}/results_and_report", "zip", pack)
print("written", zpath)
if IN_COLAB:
    from google.colab import files
    files.download(zpath)